<a href="https://colab.research.google.com/github/arabadzhyanzhela-collab/github-lab-2/blob/main/S12_%C2%B7_%C2%ABCrea_tu_IA%C2%BB_03_Red_contra_bosque_en_el_triaje.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Miniapp 3 · ¿Gana la red neuronal al bosque?

**Nivel:** 2/3 · **Duración:** 55 min · **Tema:** red densa para datos de tabla
**Practica:** S10 (red en Keras, curvas de aprendizaje, dropout, parada temprana) y S08 (bosque)

---

## Objetivo
Construir una **red neuronal** que decide si un paciente de urgencias es urgente y enfrentarla al **bosque aleatorio** y a la **regresión logística**, con una pregunta honesta:

> **¿Merece la pena la red?**

Además aprenderemos a:
- leer **curvas de aprendizaje** (cuándo deja de mejorar la red y si sobreajusta),
- comprobar que una red **no da siempre el mismo resultado** (por eso se repite con 3 semillas).

> **Aviso:** es un ejercicio de aprendizaje con datos inventados. **Nada de esto decide quién se atiende primero**: en urgencias reales decide siempre una persona.

## Datos
Fichero `triaje_urgencias.csv`: 4 000 pacientes **inventados**, 13 % urgentes.

| Columna | Significado |
|---|---|
| `edad` | años del paciente |
| `frecuencia_cardiaca` | pulsaciones por minuto |
| `tension_sistolica` | tensión arterial alta (mmHg) |
| `saturacion_o2` | oxígeno en sangre (%) |
| `temperatura_c` | temperatura corporal (°C) |
| `dolor_0_10` | dolor declarado (0 = nada, 10 = máximo) |
| `dolor_pecho` | 1 = tiene dolor de pecho |
| `dificultad_respirar` | 1 = le cuesta respirar |
| `llega_en_ambulancia` | 1 = llega en ambulancia |
| **`urgente`** | **objetivo**: 1 = necesitó atención inmediata |

## Índice
1. Preparar el entorno
2. Cargar y separar en tres partes
3. Los modelos de referencia
4. La red neuronal (3 semillas, con y sin dropout)
5. Curvas de aprendizaje
6. La tabla y el campeón
7. El examen (una sola vez) con el umbral de seguridad
8. La app
9. Ejercicios extra (con solución)
10. Hoja del equipo
11. Prueba
12. Conclusiones finales

**Cómo usar el cuaderno:** ejecuta las celdas en orden (`Mayús + Intro`). Si algo falla: *Entorno de ejecución → Reiniciar y ejecutar todo*. Las celdas de la red tardan algo más (entrenan 6 redes).

---
## Paso 1 · Preparar el entorno

Importamos las librerías:

- **numpy / pandas**: cálculos y tablas.
- **keras** (de TensorFlow): para construir y entrenar la red neuronal. Viene instalado en Colab.
- **train_test_split**: separar los datos.
- **StandardScaler**: escalar los números. **Las redes lo necesitan**: si una columna va de 0 a 200 (tensión) y otra de 0 a 1 (dolor de pecho), la red aprende mal.
- **Modelos de referencia**: `DummyClassifier` (el tonto), `LogisticRegression` y `RandomForestClassifier` (bosque).
- **Métricas**: AUC, AP, matriz de confusión y curva precisión–recall.

**Nota:** `TF_CPP_MIN_LOG_LEVEL = "3"` silencia los mensajes técnicos de TensorFlow, y `warnings.filterwarnings("ignore")` oculta avisos que no afectan al resultado.

In [1]:
import warnings, os; warnings.filterwarnings("ignore"); os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np, pandas as pd
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, precision_recall_curve

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


---
## Paso 2 · Cargar y separar en tres partes

Esta vez separamos en **tres** partes, no en dos:

| Parte | Tamaño | Para qué sirve |
|---|---|---|
| **Entrenar** (`Xa`) | 2 400 | los modelos aprenden aquí |
| **Validar** (`Xv`) | 600 | **elegir**: comparar modelos, decidir cuándo parar la red y fijar el umbral |
| **Examen** (`X_te`) | 1 000 | se mira **una sola vez**, al final |

**¿Por qué una parte de validación?** La red necesita un conjunto aparte para saber cuándo dejar de entrenar (parada temprana). Si usáramos el examen para eso, la nota final sería tramposa.

**Escalado:** el `StandardScaler` se ajusta **solo con los datos de entrenamiento** (`fit(Xa)`) y después se aplica a validación y examen. Así no «espiamos» los datos que no deberíamos ver.

**Nota:** `stratify` mantiene el 13 % de urgentes en las tres partes.

In [2]:
URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/triaje_urgencias.csv"
df = pd.read_csv(URL)
y = df["urgente"]; X = df.drop(columns="urgente")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)   # 75 % / 25 % examen
Xa, Xv, ya, yv = train_test_split(X_tr, y_tr, test_size=0.2, random_state=42, stratify=y_tr)  # del 75 %: 80 % entrenar / 20 % validar

esc = StandardScaler().fit(Xa)                 # las redes necesitan los números escalados
Za, Zv, Zte = esc.transform(Xa), esc.transform(Xv), esc.transform(X_te)

print(len(ya), "para entrenar ·", len(yv), "para validar ·", len(y_te), "para el examen (NO mirar)")
print(f"Urgentes: {y.mean():.1%} en total · {ya.mean():.1%} entrenar · {yv.mean():.1%} validar · {y_te.mean():.1%} examen")

2400 para entrenar · 600 para validar · 1000 para el examen (NO mirar)
Urgentes: 13.2% en total · 13.2% entrenar · 13.2% validar · 13.2% examen


### Qué deberías ver
**2 400** para entrenar, **600** para validar y **1 000** para el examen, con un 13 % de urgentes en cada parte.

### Conclusión del paso
Tenemos tres conjuntos separados antes de hacer nada. A partir de aquí **todas las decisiones se toman con la validación**; el examen queda guardado.

---
## Paso 3 · Los modelos de referencia

Antes de la red, entrenamos tres modelos más sencillos para tener con qué comparar:

1. **Nadie es urgente (tonto):** no mira los datos. Es el **suelo**: cualquier modelo útil debe superarlo.
2. **Regresión logística:** modelo lineal clásico (S07).
3. **Bosque aleatorio:** 300 árboles que votan (S08).

Los evaluamos en **validación** con dos métricas:
- **AUC:** si el modelo pone a los urgentes por encima de los no urgentes (0,50 = azar, 1 = perfecto).
- **AP** (precisión media): calidad de los avisos centrándose solo en los urgentes. Valor del azar ≈ 0,13.

**Nota:** la logística usa los datos escalados (`Za`, `Zv`); el bosque no lo necesita y usa los originales (`Xa`, `Xv`).

In [3]:
def notas(proba):
    return {"AUC": roc_auc_score(yv, proba), "AP": average_precision_score(yv, proba)}

tabla = {}
tabla["Nadie es urgente (tonto)"] = notas(DummyClassifier(strategy="prior").fit(Za, ya).predict_proba(Zv)[:, 1])
tabla["Logística"] = notas(LogisticRegression(max_iter=1000).fit(Za, ya).predict_proba(Zv)[:, 1])
bosque = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=2).fit(Xa, ya)
tabla["Bosque"] = notas(bosque.predict_proba(Xv)[:, 1])

pd.DataFrame(tabla).T.round(3)

,AUC,AP
Nadie es urgente (tonto),0.500,0.132
Logística,0.806,0.470
Bosque,0.872,0.620


### Qué deberías ver
| Modelo | AUC | AP |
|---|---|---|
| Nadie es urgente (tonto) | 0,50 | ≈ 0,13 |
| Logística | ≈ 0,81 | ≈ 0,47 |
| Bosque | ≈ 0,87 | ≈ 0,62 |

El tonto acertaría el 87 % de las veces diciendo «nadie es urgente», pero su AUC de 0,50 demuestra que no sabe nada.

### Conclusión del paso
El **bosque** pone el listón alto. La red tendrá que superar 0,87 de AUC para merecer la pena.

---
## Paso 4 · La red neuronal

### Cómo es la red
| Capa | Neuronas | Qué hace |
|---|---|---|
| Entrada | 9 | una por cada columna del paciente |
| Oculta 1 | 32 (ReLU) | combina las variables |
| Dropout | — | apaga al azar un % de neuronas al entrenar (para no memorizar) |
| Oculta 2 | 16 (ReLU) | combina otra vez |
| Dropout | — | igual que arriba |
| Salida | 1 (sigmoide) | probabilidad de urgente, entre 0 y 1 |

### Cómo se entrena
- **Función de error:** `binary_crossentropy` (la habitual para sí/no).
- **Optimizador:** `adam`.
- **Parada temprana (`EarlyStopping`):** si el error en validación no mejora en **15 épocas seguidas**, para y **recupera los mejores pesos**. Así evitamos que siga aprendiendo de memoria.

### Qué probamos
- **Dos versiones:** sin dropout (0,0) y con dropout del 30 % (0,3).
- **Tres semillas** para cada una (0, 1, 2). Una red empieza con pesos al azar, así que **cada entrenamiento da un número algo distinto**. Por eso damos la **media ± desviación** de las 3.

**Nota:** esta celda entrena 6 redes y tarda un poco (en torno a un minuto).

In [4]:
def red(dropout, semilla):
    keras.utils.set_random_seed(semilla)                  # fija el azar para poder repetir el resultado
    m = keras.Sequential([keras.layers.Input((Xa.shape[1],)),
                          keras.layers.Dense(32, activation="relu"), keras.layers.Dropout(dropout),
                          keras.layers.Dense(16, activation="relu"), keras.layers.Dropout(dropout),
                          keras.layers.Dense(1, activation="sigmoid")])
    m.compile("adam", "binary_crossentropy")
    h = m.fit(Za, ya, validation_data=(Zv, yv), epochs=200, batch_size=64, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)])
    return m, h

redes, historias, semillas_tabla = {}, {}, []
for dropout in (0.0, 0.3):
    auc, ap = [], []
    for semilla in range(3):
        m, h = red(dropout, semilla)
        p = m.predict(Zv, verbose=0).ravel()
        auc.append(roc_auc_score(yv, p)); ap.append(average_precision_score(yv, p))
        redes[(dropout, semilla)] = m
        historias[(dropout, semilla)] = h
        semillas_tabla.append({"dropout": dropout, "semilla": semilla, "AUC": auc[-1], "AP": ap[-1],
                               "épocas entrenadas": len(h.history["loss"])})
    print(f"Red con dropout {dropout}: AUC {np.mean(auc):.3f} ± {np.std(auc):.3f} · AP {np.mean(ap):.3f} ± {np.std(ap):.3f} (3 semillas)")
    tabla[f"Red (dropout {dropout})"] = {"AUC": float(np.mean(auc)), "AP": float(np.mean(ap))}

pd.DataFrame(semillas_tabla).round(3)

Red con dropout 0.0: AUC 0.824 ± 0.006 · AP 0.564 ± 0.013 (3 semillas)
Red con dropout 0.3: AUC 0.829 ± 0.009 · AP 0.556 ± 0.008 (3 semillas)


,dropout,semilla,AUC,AP,épocas entrenadas
0,0.0,0,0.820,0.573,40
1,0.0,1,0.821,0.545,34
2,0.0,2,0.832,0.572,50
3,0.3,0,0.838,0.565,113
4,0.3,1,0.833,0.556,179
5,0.3,2,0.817,0.546,71


### Qué deberías ver
- Red sin dropout y con dropout: **AUC ≈ 0,82–0,83 · AP ≈ 0,56**.
- Entre semillas la red varía poco (± 0,01), pero **nunca da dos veces el mismo número**.
- El dropout **casi no cambia nada**.

### Conclusión del paso
- La red **supera a la logística** (0,82 frente a 0,81 de AUC; 0,56 frente a 0,47 de AP), pero **no al bosque** (0,87 y 0,62).
- El dropout no ayuda porque **no hay mucho sobreajuste que frenar**: la red es pequeña y la parada temprana ya corta a tiempo.

---
## Paso 5 · Curvas de aprendizaje (en tabla)

La **curva de aprendizaje** muestra cómo evoluciona el error en cada época:
- **Error de entrenamiento** (`loss`): suele bajar siempre.
- **Error de validación** (`val_loss`): baja al principio y, en algún momento, **deja de mejorar** o empieza a subir.

**Cómo leerla:**
- Si los dos errores bajan juntos → la red está aprendiendo.
- Si el de entrenamiento sigue bajando y el de validación sube → **sobreajuste** (la red memoriza).
- La **mejor época** es la de menor error de validación. La parada temprana espera 15 épocas más por si mejora y, si no, para y vuelve a esa época.

En vez de un gráfico, mostramos la curva **como tabla** (algunas épocas de la semilla 0) y un resumen de dónde paró cada red.

In [5]:
def curva_en_tabla(h, cada=5):
    loss, val = h.history["loss"], h.history["val_loss"]
    epocas = sorted(set(list(range(0, len(loss), cada)) + [int(np.argmin(val)), len(loss) - 1]))
    return pd.DataFrame({"época": [e + 1 for e in epocas],
                         "error entrenamiento": [loss[e] for e in epocas],
                         "error validación": [val[e] for e in epocas]}).round(4).set_index("época")

print("Curva de aprendizaje · red SIN dropout · semilla 0")
curva_en_tabla(historias[(0.0, 0)])

Curva de aprendizaje · red SIN dropout · semilla 0


,error entrenamiento,error validación
época,,
1,0.6828,0.5606
6,0.3258,0.3147
11,0.2991,0.3006
16,0.2788,0.2914
21,0.2640,0.2874
25,0.2560,0.2863
26,0.2544,0.2864
31,0.2469,0.2871
36,0.2413,0.2884


In [6]:
print("Curva de aprendizaje · red CON dropout 0,3 · semilla 0")
curva_en_tabla(historias[(0.3, 0)])

Curva de aprendizaje · red CON dropout 0,3 · semilla 0


,error entrenamiento,error validación
época,,
1,0.6159,0.5043
6,0.3774,0.3314
11,0.3549,0.3161
16,0.3336,0.3090
21,0.3219,0.3042
26,0.3232,0.3014
31,0.3113,0.2975
36,0.3094,0.2946
41,0.3012,0.2921


In [7]:
# Resumen: dónde estuvo la mejor época y cuándo paró cada una de las 6 redes
filas = []
for (dropout, semilla), h in historias.items():
    val = h.history["val_loss"]; loss = h.history["loss"]
    mejor = int(np.argmin(val))
    filas.append({"dropout": dropout, "semilla": semilla, "mejor época": mejor + 1, "paró en la época": len(loss),
                  "error entrenamiento (mejor época)": loss[mejor], "error validación (mejor época)": val[mejor],
                  "diferencia validación − entrenamiento": val[mejor] - loss[mejor]})
resumen_curvas = pd.DataFrame(filas).round(4)
resumen_curvas

,dropout,semilla,mejor época,paró en la época,error entrenamiento (mejor época),error validación (mejor época),diferencia validación − entrenamiento
0,0.0,0,25,40,0.2560,0.2863,0.0303
1,0.0,1,19,34,0.2767,0.2897,0.0129
2,0.0,2,35,50,0.2468,0.2837,0.0368
3,0.3,0,98,113,0.2821,0.2807,-0.0014
4,0.3,1,164,179,0.2701,0.2793,0.0092
5,0.3,2,56,71,0.3054,0.2932,-0.0122


### Qué deberías ver
- **Sin dropout:** el error de validación deja de mejorar pronto (mejor época entre la 19 y la 35) y la parada temprana corta 15 épocas después (entre la 34 y la 50). En la semilla 0, a partir de la época 25 el error de entrenamiento sigue bajando (0,256 → 0,238) mientras el de validación **empieza a subir** (0,286 → 0,290): es el **inicio del sobreajuste**, y la parada temprana lo corta justo ahí.
- **Con dropout:** la red aprende **más despacio** y tarda más en estancarse (mejor época entre la 56 y la 153). Su error de entrenamiento queda **igual o por encima** del de validación, porque entrena «con neuronas apagadas»: no sobreajusta.
- Al final, el error de validación de las dos versiones es **prácticamente el mismo** (≈ 0,28–0,29).

### Conclusión del paso
La red **aprende rápido y se estanca pronto**: con 2 400 pacientes y 9 columnas no hay mucho más que extraer. El dropout frena el poco sobreajuste que hay, pero el resultado final es el mismo; solo tarda más en entrenar.

---
## Paso 6 · La tabla y el campeón

Juntamos todos los modelos y los ordenamos por **AP** (la métrica más exigente con clases desequilibradas).

In [8]:
resumen = pd.DataFrame(tabla).T.sort_values("AP", ascending=False).round(3)
print(resumen)
print("\nCampeón por AP:", resumen.index[0])

                            AUC     AP
Bosque                    0.872  0.620
Red (dropout 0.0)         0.824  0.564
Red (dropout 0.3)         0.829  0.556
Logística                 0.806  0.470
Nadie es urgente (tonto)  0.500  0.132

Campeón por AP: Bosque


### Qué deberías ver
| Puesto | Modelo | AUC | AP |
|---|---|---|---|
| 1 | **Bosque** | ≈ 0,87 | ≈ 0,62 |
| 2–3 | Red (con y sin dropout) | ≈ 0,82 | ≈ 0,56 |
| 4 | Logística | ≈ 0,81 | ≈ 0,47 |
| 5 | Nadie es urgente | 0,50 | ≈ 0,13 |

### Conclusión del paso
**La red NO gana al bosque**, aunque sí a la logística. Es el resultado habitual en **datos de tabla pequeños**: los árboles aprovechan mejor columnas ordenadas y pocos miles de filas. El campeón es el **bosque**.

---
## Paso 7 · El examen: una sola vez, con el umbral de seguridad

En urgencias, que se escape un urgente es muy grave. Por eso fijamos un objetivo de seguridad: **detectar al 90 % de los urgentes** (recall = 0,90).

**Pasos:**
1. Con las probabilidades de **validación**, buscamos el **umbral más alto** que todavía detecta al 90 % de los urgentes (el que menos falsas alarmas da).
2. Aplicamos ese umbral, **sin tocarlo**, al examen.
3. Contamos urgentes detectados y falsas alarmas.

**Nota:** el código elige automáticamente el campeón de la tabla anterior. Si fuera una red, usaría la de la semilla 0.

**Cómo leer la matriz de confusión:**

| | Predice: en cola | Predice: prioridad |
|---|---|---|
| **Real: no urgente** | VN (bien) | FP (falsa alarma) |
| **Real: urgente** | FN (se escapa) | VP (detectado) |

In [9]:
campeon = bosque if resumen.index[0] == "Bosque" else None
if campeon is not None:
    proba_v, proba_te = campeon.predict_proba(Xv)[:, 1], campeon.predict_proba(X_te)[:, 1]
else:
    m = redes[(0.3, 0)] if "0.3" in resumen.index[0] else redes[(0.0, 0)]
    proba_v, proba_te = m.predict(Zv, verbose=0).ravel(), m.predict(Zte, verbose=0).ravel()

precision, recall, umbrales = precision_recall_curve(yv, proba_v)
umbral = float(umbrales[np.where(recall[:-1] >= 0.90)[0][-1]])   # umbral elegido con la VALIDACIÓN
aviso = proba_te >= umbral

vn, fp, fn, vp = confusion_matrix(y_te, aviso).ravel()
print(f"Campeón: {resumen.index[0]}")
print(f"Umbral {umbral:.2f} · urgentes detectados {vp} de {vp + fn} · falsas alarmas {fp} de {vn + fp} no urgentes · AUC {roc_auc_score(y_te, proba_te):.3f}")
pd.DataFrame([[vn, fp], [fn, vp]], index=["Real: no urgente", "Real: urgente"],
             columns=["Predice: en cola", "Predice: prioridad"])

Campeón: Bosque
Umbral 0.06 · urgentes detectados 119 de 132 · falsas alarmas 323 de 868 no urgentes · AUC 0.887


,Predice: en cola,Predice: prioridad
Real: no urgente,545,323
Real: urgente,13,119


In [10]:
# Para comparar: ¿qué habría pasado en el examen con la red (sin dropout, semilla 0) y su propio umbral?
m_red = redes[(0.0, 0)]
pv_red, pte_red = m_red.predict(Zv, verbose=0).ravel(), m_red.predict(Zte, verbose=0).ravel()
pr, rc, us = precision_recall_curve(yv, pv_red)
umbral_red = float(us[np.where(rc[:-1] >= 0.90)[0][-1]])
vn_r, fp_r, fn_r, vp_r = confusion_matrix(y_te, pte_red >= umbral_red).ravel()
print(f"Red · umbral {umbral_red:.2f} · urgentes detectados {vp_r} de {vp_r + fn_r} · falsas alarmas {fp_r} · AUC {roc_auc_score(y_te, pte_red):.3f}")

Red · umbral 0.03 · urgentes detectados 121 de 132 · falsas alarmas 623 · AUC 0.804


### Qué deberías ver
- Bosque: con un umbral ≈ 0,06 detectas **119 de 132 urgentes (90 %)** a cambio de **≈ 320 falsas alarmas** de 868 no urgentes.
- La red, con su propio umbral (≈ 0,03), detecta 121 de 132 pero con **≈ 620 falsas alarmas**, casi el doble que el bosque, y AUC ≈ 0,80.

### Conclusión del paso
El umbral elegido con la validación **funciona también en el examen** (se mantiene el 90 %). La diferencia entre bosque y red se confirma con datos que ninguno de los dos había visto.

---
## Paso 8 · La app

La función `paciente(...)`:
1. Recibe los signos vitales de **un** paciente.
2. Los pone en una tabla con **las mismas columnas y en el mismo orden** que el entrenamiento.
3. Calcula la probabilidad de urgencia con el campeón (si fuera la red, escalaría antes los datos con `esc`).
4. Si supera el umbral → **PRIORIDAD**; si no → **en cola**.

In [11]:
def paciente(edad, frecuencia_cardiaca, tension_sistolica, saturacion_o2, temperatura_c,
             dolor_0_10, dolor_pecho, dificultad_respirar, llega_en_ambulancia):
    fila = pd.DataFrame([dict(edad=edad, frecuencia_cardiaca=frecuencia_cardiaca, tension_sistolica=tension_sistolica,
                              saturacion_o2=saturacion_o2, temperatura_c=temperatura_c, dolor_0_10=dolor_0_10,
                              dolor_pecho=dolor_pecho, dificultad_respirar=dificultad_respirar,
                              llega_en_ambulancia=llega_en_ambulancia)])[X.columns]
    if campeon is not None:
        p = campeon.predict_proba(fila)[0, 1]
    else:
        p = float(m.predict(esc.transform(fila), verbose=0)[0, 0])
    return f"probabilidad de urgencia {p:.0%} → {'PRIORIDAD' if p >= umbral else 'en cola'}"

print(paciente(70, 118, 95, 89, 38.5, 6, 1, 1, 1))   # mayor, saturación baja, dolor de pecho, le cuesta respirar
print(paciente(25, 78, 118, 99, 36.6, 2, 0, 0, 0))   # joven sano
print(paciente(52, 95, 130, 95, 37.0, 5, 0, 0, 0))   # adulto con dolor moderado

probabilidad de urgencia 78% → PRIORIDAD
probabilidad de urgencia 1% → en cola
probabilidad de urgencia 2% → en cola


### Qué deberías ver
- 70 años con saturación 89 % → **≈ 78 % → PRIORIDAD**.
- Los otros dos → **≈ 1–2 % → en cola**.

### Conclusión del paso
La app ordena con sentido común: el paciente con signos de gravedad pasa delante.

---
## Paso 9 · Ejercicios extra

1. Cambia la red a **una sola capa de 8 neuronas** o a **tres capas**: ¿cambia algo?, ¿cuántos pesos tiene cada una?
2. Pasa el **bosque** por 3 semillas (`random_state`): ¿varía tanto como la red?
3. Escribe en 3 líneas **cuándo usarías la red** en un problema de tabla y cuándo no.

**Intenta cada ejercicio antes de ejecutar la solución.**

### Extra 1 · Otra red (validación, 3 semillas)
Creamos una función que construye la red con las capas que le pasemos y la entrenamos con 3 semillas. Comparamos 1 capa de 8 neuronas, 2 capas (32-16) y 3 capas (64-32-16).

**Nota:** `count_params()` cuenta los **pesos** de la red (los números que aprende). Más pesos = red más grande.

In [12]:
def red_capas(capas, semilla):
    keras.utils.set_random_seed(semilla)
    m = keras.Sequential([keras.layers.Input((Xa.shape[1],))]
                         + [keras.layers.Dense(u, activation="relu") for u in capas]
                         + [keras.layers.Dense(1, activation="sigmoid")])
    m.compile("adam", "binary_crossentropy")
    m.fit(Za, ya, validation_data=(Zv, yv), epochs=200, batch_size=64, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)])
    return m

extra1 = {}
for nombre, capas in (("1 capa de 8", (8,)), ("2 capas 32-16", (32, 16)), ("3 capas 64-32-16", (64, 32, 16))):
    auc, ap = [], []
    for semilla in range(3):
        mc = red_capas(capas, semilla)
        p = mc.predict(Zv, verbose=0).ravel()
        auc.append(roc_auc_score(yv, p)); ap.append(average_precision_score(yv, p))
    extra1[nombre] = {"AUC media": np.mean(auc), "± AUC": np.std(auc), "AP media": np.mean(ap), "pesos": mc.count_params()}
pd.DataFrame(extra1).T.round(3)

,AUC media,± AUC,AP media,pesos
1 capa de 8,0.820,0.003,0.543,89.0
2 capas 32-16,0.824,0.006,0.564,865.0
3 capas 64-32-16,0.822,0.004,0.554,3265.0


**Conclusión:**

| Red | AUC | AP | Pesos |
|---|---|---|---|
| 1 capa de 8 | ≈ 0,82 | ≈ 0,54 | 89 |
| 2 capas 32-16 | ≈ 0,82 | ≈ 0,56 | 865 |
| 3 capas 64-32-16 | ≈ 0,82 | ≈ 0,55 | 3 265 |

**No cambia nada.** Una red de 89 pesos hace lo mismo que una de 3 265: **el problema no pide más red**. Hacerla más grande solo la hace más lenta.

### Extra 2 · Bosque con 3 semillas
Entrenamos el bosque con `random_state` 0, 1 y 2 y comparamos la **variación entre semillas** con la **diferencia** entre bosque y red.

In [13]:
auc_b, ap_b = [], []
for semilla in range(3):
    b = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=semilla, n_jobs=2).fit(Xa, ya)
    p = b.predict_proba(Xv)[:, 1]
    auc_b.append(roc_auc_score(yv, p)); ap_b.append(average_precision_score(yv, p))
print(f"Bosque: AUC {np.mean(auc_b):.3f} ± {np.std(auc_b):.4f} · AP {np.mean(ap_b):.3f} ± {np.std(ap_b):.4f}")
print(f"Diferencia de AUC bosque − red (sin dropout): {np.mean(auc_b) - tabla['Red (dropout 0.0)']['AUC']:.3f}")

Bosque: AUC 0.871 ± 0.0028 · AP 0.615 ± 0.0040
Diferencia de AUC bosque − red (sin dropout): 0.046


**Conclusión:** el bosque varía **muy poco** entre semillas (± 0,003 de AUC), incluso menos que la red. Lo importante: la **diferencia bosque–red (≈ 0,05 de AUC)** es **mucho mayor** que cualquier variación entre semillas. Por tanto, **el bosque gana de verdad, no por azar**.

### Extra 3 · Cuándo usar la red en datos de tabla
*(Respuesta de redacción: no hay una única respuesta correcta.)*

**Nuestra respuesta:**

> …

**Respuesta modelo:**
- **Sí usaría la red** con muchísimos datos, con tablas **mezcladas con imágenes o texto**, o si el modelo tiene que **seguir aprendiendo** con datos nuevos.
- **No la usaría** con pocos datos (unos miles) y columnas ordenadas: **el bosque o el boosting** ganan, entrenan en segundos y se explican mejor.
- En este caso, el bosque gana por ≈ 0,05 de AUC y la red no aporta nada que compense su complejidad.

---
## Paso 10 · Hoja del equipo

| Campo | Contenido |
|---|---|
| **Equipo** | *(nombres)* |
| **Miniapp** | 3 · ¿Gana la red neuronal al bosque? |
| **Fecha** | *(dd/mm/aaaa)* |
| **Datos** | 4 000 pacientes inventados · 13 % urgentes |
| **Separación** | 2 400 entrenar · 600 validar · 1 000 examen (estratificada) |
| **Modelo de referencia** | «Nadie es urgente»: AUC 0,50 |
| **Modelos comparados** | Logística, bosque, red 32-16 sin dropout y con dropout 0,3 (3 semillas cada una) |
| **Campeón** | **Bosque** (AUC ≈ 0,87 · AP ≈ 0,62 en validación) |
| **¿Ganó la red?** | **No.** AUC ≈ 0,82 · AP ≈ 0,56: gana a la logística, pierde con el bosque |
| **Curvas de aprendizaje** | Se estanca en pocas decenas de épocas; poco sobreajuste; el dropout no cambia nada |
| **Umbral** | ≈ 0,06, elegido con validación para detectar al 90 % de los urgentes |
| **Resultado en el examen** | 119 de 132 urgentes detectados · ≈ 320 falsas alarmas de 868 |

### Tarjeta de cierre
- **Mi IA decide** si un paciente es urgente **a partir de** dolor, pulso, saturación, respiración y el resto de signos vitales.
- **Se equivoca** con 13 de 132 urgentes en el examen (y da ≈ 320 falsas alarmas), **frente a** 132 de 132 del modelo de referencia.
- **Falla cuando** el paciente es urgente pero tiene síntomas poco llamativos.
- **No debe usarse para** decidir sin una persona.

### Decisión del equipo: ¿mereció la pena la red?
*(2–3 frases con vuestras palabras.)*

> …

### Lo que hemos aprendido
- *(2–3 ideas)*

La celda siguiente imprime **vuestras cifras reales** para comprobar o completar la tabla:

In [14]:
r0 = tabla["Red (dropout 0.0)"]; r3 = tabla["Red (dropout 0.3)"]
hoja = pd.Series({
    "Entrenar / validar / examen": f"{len(ya)} / {len(yv)} / {len(y_te)}",
    "AUC / AP logística": f"{tabla['Logística']['AUC']:.3f} / {tabla['Logística']['AP']:.3f}",
    "AUC / AP bosque": f"{tabla['Bosque']['AUC']:.3f} / {tabla['Bosque']['AP']:.3f}",
    "AUC / AP red sin dropout (media 3 semillas)": f"{r0['AUC']:.3f} / {r0['AP']:.3f}",
    "AUC / AP red con dropout (media 3 semillas)": f"{r3['AUC']:.3f} / {r3['AP']:.3f}",
    "Mejor época (red sin dropout, semilla 0)": int(resumen_curvas.iloc[0]["mejor época"]),
    "Campeón": resumen.index[0],
    "Umbral": f"{umbral:.2f}",
    "Urgentes detectados en examen": f"{vp} de {vp + fn}",
    "Falsas alarmas en examen": f"{fp} de {vn + fp}",
    "AUC en examen": f"{roc_auc_score(y_te, proba_te):.3f}",
}, name="Valor")
hoja.to_frame()

,Valor
Entrenar / validar / examen,2400 / 600 / 1000
AUC / AP logística,0.806 / 0.470
AUC / AP bosque,0.872 / 0.620
AUC / AP red sin dropout (media 3 semillas),0.824 / 0.564
AUC / AP red con dropout (media 3 semillas),0.829 / 0.556
"Mejor época (red sin dropout, semilla 0)",25
Campeón,Bosque
Umbral,0.06
Urgentes detectados en examen,119 de 132
Falsas alarmas en examen,323 de 868


---
## Paso 11 · Prueba

### A) Formulario para probar un paciente
En Colab, a la derecha de la celda aparecen **casillas y deslizadores**. Cambia los valores y vuelve a ejecutar la celda.

**Nota:** las líneas con `#@param` son las que Colab convierte en formulario. Fuera de Colab funcionan como variables normales.

In [15]:
#@title Revisar un paciente
edad = 60  #@param {type:"slider", min:0, max:100, step:1}
frecuencia_cardiaca = 105  #@param {type:"slider", min:40, max:150, step:1}
tension_sistolica = 110  #@param {type:"slider", min:60, max:200, step:1}
saturacion_o2 = 92  #@param {type:"slider", min:85, max:100, step:1}
temperatura_c = 37.8  #@param {type:"number"}
dolor_0_10 = 5  #@param {type:"slider", min:0, max:10, step:1}
dolor_pecho = 0  #@param [0, 1]
dificultad_respirar = 1  #@param [0, 1]
llega_en_ambulancia = 0  #@param [0, 1]

print(paciente(edad, frecuencia_cardiaca, tension_sistolica, saturacion_o2, temperatura_c,
               dolor_0_10, dolor_pecho, dificultad_respirar, llega_en_ambulancia))

probabilidad de urgencia 81% → PRIORIDAD


### B) Nuestros 3 casos (los del README)
El decálogo pide **probar la app con 3 casos propios y contar dónde falla**. Usamos:
1. Un paciente claramente **grave** inventado por nosotros.
2. Un paciente claramente **sano** inventado por nosotros.
3. Un paciente **real del examen** que era urgente y al que el bosque dejó en cola, para ver **dónde falla**.

La tercera celda busca automáticamente ese paciente entre los urgentes que se escaparon en el examen.

In [16]:
print("Caso 1 · Asmático en crisis (35 años, saturación 90 %, pulso 120, le cuesta respirar):")
print("  ", paciente(35, 120, 125, 90, 37.1, 3, 0, 1, 0))
print("Caso 2 · Control rutinario (50 años, constantes normales, dolor 1):")
print("  ", paciente(50, 72, 128, 98, 36.5, 1, 0, 0, 0))

Caso 1 · Asmático en crisis (35 años, saturación 90 %, pulso 120, le cuesta respirar):
   probabilidad de urgencia 74% → PRIORIDAD
Caso 2 · Control rutinario (50 años, constantes normales, dolor 1):
   probabilidad de urgencia 10% → PRIORIDAD


In [17]:
# Caso 3: urgentes reales del examen que el campeón dejó en cola (falsos negativos)
escapados = X_te[(y_te.values == 1) & ~aviso].copy()
escapados["prob_urgente"] = proba_te[(y_te.values == 1) & ~aviso].round(3)
print(f"Urgentes que se escaparon en el examen: {len(escapados)}")
print("\nMedia de los urgentes que se ESCAPARON frente a los DETECTADOS:")
detectados = X_te[(y_te.values == 1) & aviso]
pd.DataFrame({"escapados": escapados[X.columns].mean(), "detectados": detectados.mean()}).round(1)

Urgentes que se escaparon en el examen: 13

Media de los urgentes que se ESCAPARON frente a los DETECTADOS:


,escapados,detectados
edad,46.7,53.0
frecuencia_cardiaca,79.7,94.4
tension_sistolica,121.1,128.5
saturacion_o2,96.6,95.5
temperatura_c,36.8,37.0
dolor_0_10,2.7,5.1
dolor_pecho,0.2,0.2
dificultad_respirar,0.0,0.3
llega_en_ambulancia,0.2,0.3


In [18]:
caso3 = escapados.sort_values("prob_urgente").iloc[0]      # el urgente con menos probabilidad
print("Caso 3 · Urgente REAL del examen que el modelo deja en cola:")
print(caso3[X.columns].to_dict())
print("  ", paciente(*[caso3[c] for c in X.columns]), "· en realidad ERA urgente")

Caso 3 · Urgente REAL del examen que el modelo deja en cola:
{'edad': 54.0, 'frecuencia_cardiaca': 90.0, 'tension_sistolica': 129.0, 'saturacion_o2': 94.0, 'temperatura_c': 37.7, 'dolor_0_10': 2.0, 'dolor_pecho': 0.0, 'dificultad_respirar': 0.0, 'llega_en_ambulancia': 0.0}
   probabilidad de urgencia 1% → en cola · en realidad ERA urgente


### Qué deberías ver
- **Caso 1 (asmático):** ≈ 74 % → PRIORIDAD → **acierta**.
- **Caso 2 (control rutinario):** ≈ 10 % → PRIORIDAD → **falla (falsa alarma)**. Como el umbral es muy bajo (≈ 0,06) para no dejar escapar urgentes, avisa incluso con probabilidades pequeñas.
- **Caso 3 (urgente real del examen):** 54 años, dolor 2, sin dificultad para respirar ni dolor de pecho → ≈ 1 % → en cola → **falla (urgente que se escapa)**.

**Dónde falla:** los urgentes que se escapan tienen, de media, **menos dolor** (2,7 frente a 5,1), **ninguno tiene dificultad para respirar** y su pulso es **normal** (80 frente a 94). Son urgentes «silenciosos»: con estas 9 columnas el modelo no tiene ninguna señal de gravedad. El fallo contrario (caso 2) es el precio del umbral de seguridad.

### C) Comprobaciones automáticas
Cada línea indica **OK** o **FALLO**.

In [19]:
def check(condicion, mensaje):
    print(("OK    " if condicion else "FALLO ") + mensaje)
    return bool(condicion)

ok = all([
    check((len(ya), len(yv), len(y_te)) == (2400, 600, 1000), "Separación 2 400 / 600 / 1 000"),
    check(abs(tabla["Nadie es urgente (tonto)"]["AUC"] - 0.5) < 1e-9, "El modelo tonto tiene AUC 0,50"),
    check(tabla["Bosque"]["AUC"] > tabla["Red (dropout 0.0)"]["AUC"], "El bosque supera a la red en AUC"),
    check(tabla["Red (dropout 0.0)"]["AP"] > tabla["Logística"]["AP"], "La red supera a la logística en AP"),
    check(abs(tabla["Red (dropout 0.0)"]["AUC"] - tabla["Red (dropout 0.3)"]["AUC"]) < 0.02, "El dropout casi no cambia la AUC"),
    check(resumen_curvas["paró en la época"].max() < 200, "La parada temprana cortó todas las redes antes de 200 épocas"),
    check(vp / (vp + fn) >= 0.85, "En el examen se detecta al menos el 85 % de los urgentes"),
    check("PRIORIDAD" in paciente(70, 118, 95, 89, 38.5, 6, 1, 1, 1), "El paciente grave recibe PRIORIDAD"),
    check("en cola" in paciente(25, 78, 118, 99, 36.6, 2, 0, 0, 0), "El joven sano queda en cola"),
])
print("\nTodas las pruebas superadas." if ok else "\nRevisa las pruebas marcadas como FALLO.")

OK    Separación 2 400 / 600 / 1 000
OK    El modelo tonto tiene AUC 0,50
OK    El bosque supera a la red en AUC
OK    La red supera a la logística en AP
OK    El dropout casi no cambia la AUC
OK    La parada temprana cortó todas las redes antes de 200 épocas
OK    En el examen se detecta al menos el 85 % de los urgentes
OK    El paciente grave recibe PRIORIDAD
OK    El joven sano queda en cola

Todas las pruebas superadas.


---
## Conclusiones finales

1. **Una red no siempre gana.** En esta tabla pequeña (2 400 pacientes, 9 columnas), el **bosque** supera a la red por ≈ 0,05 de AUC. La red solo gana a la logística.
2. **La diferencia es real, no azar:** la variación entre semillas (± 0,01 la red, ± 0,003 el bosque) es mucho menor que la diferencia entre modelos.
3. **Una red no da dos veces el mismo número:** hay que fijar la semilla, repetir varias veces y dar la media.
4. **Las curvas de aprendizaje** muestran que la red se estanca pronto y apenas sobreajusta; por eso **el dropout no aporta nada** y una red de 89 pesos hace lo mismo que una de 3 265.
5. **El umbral se decide con la validación**, pensando en la seguridad: detectar al 90 % de los urgentes cuesta ≈ 320 falsas alarmas de 868.
6. **El modelo falla con los urgentes «silenciosos»** y, en cualquier caso, **avisa pero no decide**: siempre con una persona.